In [ ]:
# Optional: sanity check GPU
!nvidia-smi

Thu Nov 13 07:18:48 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

#Cell 1 — Mount Drive, paths, helpers

In [ ]:
# ============================
# Mount Drive, paths, helpers
# ============================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil, subprocess, shlex, os, json, gc, time

# ---------- Drive roots (EDIT if needed) ----------
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/ft_vs_rag_project")
DRIVE_DATA_DIR    = DRIVE_PROJECT_DIR / "ft_vs_rag_multidata" / "data_hotpot"
DRIVE_IDX_FULL    = DRIVE_PROJECT_DIR / "ft_vs_rag_multidata" / "faiss_index_full"
DRIVE_IDX_DYN     = DRIVE_PROJECT_DIR / "ft_vs_rag_multidata" / "faiss_index_dynamic"
DRIVE_OUT_DIR     = DRIVE_PROJECT_DIR / "notebook_outputs"      # where we will copy final artifacts
DRIVE_REPORTS_DIR = DRIVE_PROJECT_DIR / "reports"               # optional reports

# ---------- Local working dirs (fast, ephemeral) ----------
LOCAL_ROOT     = Path("/content/ft_vs_rag_work")
LOCAL_DATA_DIR = LOCAL_ROOT / "data"
LOCAL_IDX_DIR  = LOCAL_ROOT / "faiss_indexes"
LOCAL_OUT_DIR  = LOCAL_ROOT / "outputs"
LOCAL_LOG_DIR  = LOCAL_ROOT / "logs"
LOCAL_TMP_DIR  = LOCAL_ROOT / "tmp"
LOCAL_ZIP_DIR  = LOCAL_ROOT / "zips"

for p in [LOCAL_DATA_DIR, LOCAL_IDX_DIR, LOCAL_OUT_DIR, LOCAL_LOG_DIR, LOCAL_TMP_DIR, LOCAL_ZIP_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Drive root:", DRIVE_PROJECT_DIR)
print("Local root:", LOCAL_ROOT)

# ---------- Helpers ----------
def rsync_dir(src: Path, dst: Path):
    """Efficient directory sync (copies only what changed)."""
    dst.mkdir(parents=True, exist_ok=True)
    cmd = f'rsync -ah --info=NAME1,STATS1 "{src.as_posix()}/" "{dst.as_posix()}/"'
    print(">>", cmd)
    subprocess.check_call(shlex.split(cmd))

def safe_copy_file(src: Path, dst: Path):
    """Atomic single-file copy (reduces chance of partial writes)."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_suffix(dst.suffix + ".tmp")
    shutil.copy2(src, tmp)
    tmp.replace(dst)

def zip_dir(src: Path, dst_zip: Path):
    """Zip a directory into dst_zip."""
    dst_zip.parent.mkdir(parents=True, exist_ok=True)
    if dst_zip.exists():
        dst_zip.unlink()
    cmd = f'cd "{src.parent.as_posix()}" && zip -r "{dst_zip.name}" "{src.name}"'
    print(">>", cmd)
    subprocess.check_call(shlex.split(cmd))


Mounted at /content/drive
Drive root: /content/drive/MyDrive/ft_vs_rag_project
Local root: /content/ft_vs_rag_work


#Cell 2 — Install deps (GPU-safe), set constants

In [ ]:
# ============================
# Installs & constants
# ============================
!pip -q install sentence-transformers faiss-cpu

# Retrieval encoder MUST match the index that was built (mpnet → 768-d)
MODEL_NAME   = "sentence-transformers/all-mpnet-base-v2"
MAX_Q_TEST   = 200      # number of questions to evaluate in Hit@K (you can increase later)
TOP_K        = 5
BATCH_SIZE   = 16       # reduce to 8 if VRAM is tight (T4)
MAX_SEQ_LEN  = 256      # shorter input length for encoder → less VRAM, keeps retrieval quality strong

# CUDA memory hygiene
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 56.1 MB/s eta 0:00:00


#Cell 3 — Copy inputs Drive → Local (datasets + FAISS)

In [ ]:
# ============================
# Copy inputs Drive → Local
# ============================

# Datasets required for this notebook
INPUT_FILES = [
    "docs_for_pretrain.jsonl",       # (may be used for stats or sanity)
    "docs_qa_with_context.jsonl",    # SFT QA+context we may re-create or evaluate
    "test_qas_unseen.jsonl",         # evaluation set for strict retrieval tests
    "stable_docs.jsonl",             # for title filtering / sanity checks
    "dynamic_test_docs.jsonl",        # for leakage checks
    "train_ft_qas.jsonl"              # for training SML on qa
]

for name in INPUT_FILES:
    src = DRIVE_DATA_DIR / name
    dst = LOCAL_DATA_DIR / name
    assert src.exists(), f"Missing on Drive: {src}"
    shutil.copy2(src, dst)
    print("Copied:", src, "->", dst)

# Copy FAISS indexes (to avoid Drive I/O while retrieving)
rsync_dir(DRIVE_IDX_FULL, LOCAL_IDX_DIR / "faiss_index_full")
rsync_dir(DRIVE_IDX_DYN,  LOCAL_IDX_DIR / "faiss_index_dynamic")

# Local paths for the rest of the notebook
DATA_ROOT = LOCAL_DATA_DIR
IDX_FULL  = LOCAL_IDX_DIR / "faiss_index_full"
IDX_DYN   = LOCAL_IDX_DIR / "faiss_index_dynamic"

# Sanity
assert (IDX_FULL / "index.faiss").exists() and (IDX_FULL / "meta.json").exists()
assert (IDX_DYN  / "index.faiss").exists() and (IDX_DYN  / "meta.json").exists()
print("Local data:", sorted(x.name for x in LOCAL_DATA_DIR.glob("*.jsonl")))
print("Local FAISS (FULL):", sorted(x.name for x in (IDX_FULL).glob("*"))[:5])
print("Local FAISS (DYN): ", sorted(x.name for x in (IDX_DYN).glob("*"))[:5])


Copied: /content/drive/MyDrive/ft_vs_rag_project/ft_vs_rag_multidata/data_hotpot/docs_for_pretrain.jsonl -> /content/ft_vs_rag_work/data/docs_for_pretrain.jsonl
Copied: /content/drive/MyDrive/ft_vs_rag_project/ft_vs_rag_multidata/data_hotpot/docs_qa_with_context.jsonl -> /content/ft_vs_rag_work/data/docs_qa_with_context.jsonl
Copied: /content/drive/MyDrive/ft_vs_rag_project/ft_vs_rag_multidata/data_hotpot/test_qas_unseen.jsonl -> /content/ft_vs_rag_work/data/test_qas_unseen.jsonl
Copied: /content/drive/MyDrive/ft_vs_rag_project/ft_vs_rag_multidata/data_hotpot/stable_docs.jsonl -> /content/ft_vs_rag_work/data/stable_docs.jsonl
Copied: /content/drive/MyDrive/ft_vs_rag_project/ft_vs_rag_multidata/data_hotpot/dynamic_test_docs.jsonl -> /content/ft_vs_rag_work/data/dynamic_test_docs.jsonl
Copied: /content/drive/MyDrive/ft_vs_rag_project/ft_vs_rag_multidata/data_hotpot/train_ft_qas.jsonl -> /content/ft_vs_rag_work/data/train_ft_qas.jsonl
>> rsync -ah --info=NAME1,STATS1 "/content/drive/MyDri

#Cell 4 — Shared utilities (I/O + retrieval helpers)

In [ ]:
# ============================
# Shared utilities
# ============================
from sentence_transformers import SentenceTransformer
import numpy as np, faiss, torch

# One shared CUDA encoder (GPU), with small memory footprint
torch.set_grad_enabled(False)
encoder = SentenceTransformer(MODEL_NAME, device="cuda")
encoder.max_seq_length = MAX_SEQ_LEN  # VRAM-friendly
print("Encoder:", MODEL_NAME, "| device:", encoder._target_device, "| max_seq_len:", encoder.max_seq_length)

def clear_gpu():
    torch.cuda.empty_cache(); gc.collect()

def read_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

def write_jsonl(path: Path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def titles_from_docs(docs):
    return {(d.get("title") or "").strip() for d in docs if d.get("title")}

def load_meta(idx_dir: Path):
    return read_jsonl(idx_dir / "meta.json")

def faiss_search(index_dir: Path, q_embs_np: np.ndarray, top_k: int):
    """FAISS runs on CPU; embeddings are numpy on CPU already."""
    idx = faiss.read_index(str(index_dir / "index.faiss"))
    D, I = idx.search(q_embs_np.astype("float32"), top_k)
    return D, I

def encode_questions_gpu(q_texts, batch_size=BATCH_SIZE, normalize=True):
    """Encode queries on GPU in small batches; return numpy on CPU."""
    if not q_texts:
        return np.zeros((0, 768), dtype="float32")
    chunks = []
    with torch.inference_mode(), torch.amp.autocast("cuda", dtype=torch.float16):
        for s in range(0, len(q_texts), batch_size):
            batch = q_texts[s:s+batch_size]
            q_embs = encoder.encode(
                batch,
                batch_size=len(batch),
                show_progress_bar=False,
                convert_to_numpy=True,
                normalize_embeddings=normalize
            )
            chunks.append(q_embs)
            clear_gpu()
    return np.vstack(chunks)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoder: sentence-transformers/all-mpnet-base-v2 | device: cuda:0 | max_seq_len: 256


#Cell 5 — RAG tests (integrity, leakage, manual smoke, Hit@K)

In [ ]:
# ============================
# RAG validation tests
# ============================

# --- TEST 1: index integrity ---
print("\n=== TEST 1: index integrity ===")
print("DYNAMIC meta rows:", len(load_meta(IDX_DYN)))
print("FULL    meta rows:", len(load_meta(IDX_FULL)))

# --- TEST 2: leakage guard (held-out dynamic titles must NOT be in any index) ---
print("\n=== TEST 2: leakage guard ===")
heldout_titles = titles_from_docs(read_jsonl(DATA_ROOT / "dynamic_test_docs.jsonl"))
for label, idx_dir in [("HYBRID", IDX_DYN), ("FULL", IDX_FULL)]:
    idx_titles = titles_from_docs(load_meta(idx_dir))
    overlap = heldout_titles & idx_titles
    assert not overlap, f"[{label}] Leakage: held-out titles present in index (e.g. {list(sorted(overlap))[:5]})"
    print(f"OK: [{label}] no held-out titles in index")

# --- TEST 3: manual smoke queries ---
def retrieve_context_gpu(question: str, idx_dir: Path, k: int = TOP_K) -> str:
    q_embs = encode_questions_gpu([question], batch_size=1, normalize=True)
    _, I = faiss_search(idx_dir, q_embs, k)
    meta = load_meta(idx_dir)
    hits = [meta[j] for j in I[0] if 0 <= j < len(meta)]
    return "\n\n".join(h.get("text","") for h in hits)

print("\n=== TEST 3: manual retrieval smoke (5 examples) ===")
examples = [
    "Who was the first person to ...?",
    "Which book was written by ...?",
    "What is the capital of the country mentioned with ...?",
    "Which two entities are connected by ...?",
    "When did the event related to ... occur?"
]
for q in examples:
    ctx_h = retrieve_context_gpu(q, IDX_DYN,  k=TOP_K)
    ctx_f = retrieve_context_gpu(q, IDX_FULL, k=TOP_K)
    print(f"\nQ: {q}\n[hybrid]\n{ctx_h[:280].replace('\\n',' ')}...")
    print(f"[full]\n{ctx_f[:280].replace('\\n',' ')}...")

# --- TEST 4: Title Hit@K on strict unseen set ---
def title_hit_at_k_gpu(qafile: Path, idx_dir: Path, k: int = TOP_K, max_q: int = MAX_Q_TEST, batch_size: int = BATCH_SIZE):
    qas = read_jsonl(qafile)[:max_q]
    q_texts, supp_list = [], []
    for q in qas:
        supp = {(t or '').strip() for t in q.get("supporting_docs", []) if t}
        if not supp:
            continue
        q_texts.append(q["question"])
        supp_list.append(supp)

    if not q_texts:
        print("No questions with supporting docs in file.")
        return 0.0

    q_embs = encode_questions_gpu(q_texts, batch_size=batch_size, normalize=True)
    _, I = faiss_search(idx_dir, q_embs, k)
    meta = load_meta(idx_dir)

    hits = 0
    for i, supp in enumerate(supp_list):
        ret_titles = {(meta[j].get("title") or '').strip() for j in I[i] if 0 <= j < len(meta)}
        if supp & ret_titles:
            hits += 1

    score = hits / len(supp_list)
    print(f"Title Hit@{k}: {hits}/{len(supp_list)} = {score:.3f}")
    return score

print("\n=== TEST 4: Title Hit@5 on strict unseen set ===")
qafile = DATA_ROOT / "test_qas_unseen.jsonl"
print("[HYBRID] (dynamic-only index):")
title_hit_at_k_gpu(qafile, IDX_DYN,  k=TOP_K, max_q=MAX_Q_TEST, batch_size=BATCH_SIZE)
print("[FULL] (stable + dynamic-indexed):")
title_hit_at_k_gpu(qafile, IDX_FULL, k=TOP_K, max_q=MAX_Q_TEST, batch_size=BATCH_SIZE)



=== TEST 1: index integrity ===
DYNAMIC meta rows: 20384
FULL    meta rows: 553739

=== TEST 2: leakage guard ===
OK: [HYBRID] no held-out titles in index
OK: [FULL] no held-out titles in index

=== TEST 3: manual retrieval smoke (5 examples) ===

Q: Who was the first person to ...?
[hybrid]
Captain Albert Berry is one of two people credited as the first person to make a successful parachute jump from a powered airplane.  The other contender is Grant Morton, who is reported to have jumped from a Wright Model B flying over Venice Beach, California sometime late in 191...
[full]
The Man Who Invented the Computer is a 2010 historical biography by author Jane Smiley about American physicist John Vincent Atanasoff and the invention of the computer.  The book follows Atanasoff as he collaborates with others to develop the Atanasoff-Berry Computer (ABC), the ...

Q: Which book was written by ...?
[hybrid]
Desmond Elliott (1930 – 2003) was a distinguished publisher and literary agent.  Having

0.0

#Cell 6 — Build SFT QA+context dataset (GPU encoder), write locally

In [ ]:
# ============================
# Build SFT QA+context dataset (locally)
# ============================
from typing import Optional, Set

def create_qa_finetune_dataset_from_index_gpu(
    qas_file: Path,
    index_dir: Path,
    out_file: Path,
    top_k: int = 3,
    allowed_titles: Optional[Set[str]] = None,
    search_k: int = 25,
    batch_size: int = BATCH_SIZE
) -> Path:
    """
    Reads QAs, retrieves contexts via FAISS, writes JSONL:
      {"id","question","answer","context"}.
    Uses GPU encoder for questions.
    If allowed_titles provided, filter results to that set (search_k >= top_k).
    """
    qas = read_jsonl(qas_file)
    qs, ids, ans = [], [], []
    for q in qas:
        qs.append(q.get("question",""))
        ids.append(q.get("id"))
        ans.append(q.get("answer",""))

    print(f"Loaded {len(qs)} QAs. Encoding on GPU...")
    q_embs = encode_questions_gpu(qs, batch_size=batch_size, normalize=True)

    print("FAISS batched search...")
    _, I = faiss_search(index_dir, q_embs, search_k)
    meta = load_meta(index_dir)

    out_rows = []
    for i in range(len(qs)):
        hits = [meta[j] for j in I[i] if 0 <= j < len(meta)]
        if allowed_titles is not None:
            hits = [h for h in hits if (h.get("title") or "").strip() in allowed_titles]
        hits = hits[:top_k]
        if not hits:
            continue
        context = "\n\n".join(h.get("text","") for h in hits)
        out_rows.append({"id": ids[i], "question": qs[i], "answer": ans[i], "context": context})

    write_jsonl(out_file, out_rows)
    print(f"Wrote {out_file} : {len(out_rows)} rows")
    return out_file

# (Recommended) Stable-only SFT set: filter to stable titles while retrieving from FULL index
stable_titles = titles_from_docs(read_jsonl(DATA_ROOT / "stable_docs.jsonl"))
SFT_LOCAL = LOCAL_DATA_DIR / "docs_qa_with_context_STABLEONLY.jsonl"

_ = create_qa_finetune_dataset_from_index_gpu(
    qas_file   = DATA_ROOT / "train_ft_qas.jsonl",
    index_dir  = IDX_FULL,          # use full index; filter to stable only
    out_file   = SFT_LOCAL,         # write locally
    top_k      = 3,
    allowed_titles = stable_titles, # ensure FT learns on stable domain
    search_k   = 25,
    batch_size = BATCH_SIZE
)


Loaded 90447 QAs. Encoding on GPU...
FAISS batched search...
Wrote /content/ft_vs_rag_work/data/docs_qa_with_context_STABLEONLY.jsonl : 90447 rows


#Cell 7 — (Optional) Quick stats / sample preview

In [ ]:
# ============================
# Quick stats / preview
# ============================
def file_head(path: Path, n=3):
    rows = read_jsonl(path)[:n]
    for i, r in enumerate(rows):
        print(f"\n[{i}] id={r.get('id')}")
        print("Q:", r.get("question","")[:200])
        print("A:", r.get("answer","")[:200])
        print("CTX:", (r.get("context","")[:300]).replace("\n"," ") + ("..." if len(r.get("context",""))>300 else ""))

print("Preview of SFT set:")
file_head(SFT_LOCAL, n=3)


Preview of SFT set:

[0] id=5a7a06935542990198eaf050
Q: Which magazine was started first Arthur's Magazine or First for Women?
A: Arthur's Magazine
CTX: Arthur magazine was a bi-monthly periodical that was founded in October 2002, by publisher Laris Kreslins and editor Jay Babcock.  It received favorable attention from other periodicals such as "L.A. Weekly", "Print", "Punk Planet" and "Rolling Stone".  "Arthur" featured photography and artwork from...

[1] id=5a879ab05542996e4f30887e
Q: The Oberoi family is part of a hotel company that has a head office in what city?
A: Delhi
CTX: The Oberoi Group is a hotel company with its head office in Delhi.  Founded in 1934, the company owns and/or operates 30+ luxury hotels and two river cruise ships in six countries, primarily under its Oberoi Hotels & Resorts and Trident Hotels brands. The Oberoi Group is a hotel company with its hea...

[2] id=5a8d7341554299441c6b9fe5
Q: Musician and satirist Allie Goertz wrote a song about the "The Simpsons

#Cell 8 — Save results: zip + copy Local → Drive (one shot)

In [ ]:
# ============================
# End-of-run save: Local → Drive
# ============================

# 1) Zip large folders locally (if any). For this notebook we mostly wrote files, not heavy dirs.
#    If you produced large dirs (e.g., model adapters), add them to the list below.
to_zip_dirs = []  # e.g., [LOCAL_OUT_DIR / "mistral_stage2"]

for d in to_zip_dirs:
    if d.exists():
        z = LOCAL_ZIP_DIR / f"{d.name}.zip"
        zip_dir(d, z)

# 2) Copy outputs to Drive (atomic copy for files; rsync for folders)
DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Copy SFT dataset(s) and any artifacts you want to persist
artifacts = [
    SFT_LOCAL,
    LOCAL_DATA_DIR / "docs_qa_with_context.jsonl",                # if you also generated a non-filtered variant
    LOCAL_OUT_DIR / "rag_validation_report.json"                  # if you wrote a report
]
for a in artifacts:
    if a.exists():
        safe_copy_file(a, DRIVE_OUT_DIR / a.name)
        print("Copied to Drive:", DRIVE_OUT_DIR / a.name)

# Copy zipped dirs (if any)
for z in LOCAL_ZIP_DIR.glob("*.zip"):
    safe_copy_file(z, DRIVE_OUT_DIR / z.name)
    print("Copied zip to Drive:", DRIVE_OUT_DIR / z.name)

print("\n✅ Done. All outputs written locally and copied to Drive at the end.")


Copied to Drive: /content/drive/MyDrive/ft_vs_rag_project/notebook_outputs/docs_qa_with_context_STABLEONLY.jsonl
Copied to Drive: /content/drive/MyDrive/ft_vs_rag_project/notebook_outputs/docs_qa_with_context.jsonl

✅ Done. All outputs written locally and copied to Drive at the end.
